# CS 521 HW3, Problem 1

Use Google Colab with a GPU: **Runtime > Change runtime type > T4 GPU**.

All the code is in this notebook. Complete each place that has a `TODO` and `raise NotImplementedError`.
Do not change the data, the model, the attacks, the evaluation or the default hyperparameters,
unless the homework tells you to change them.

**Threat model.** For each test image $x$, the perturbation region is
$B_\infty(x, 8/255) \cup B_2(x, 0.5) \cup B_1(x, 12)$.
The union accuracy is the percentage of test images that are robust against all three attacks simultaneously.

# Set up for dataset and model

Download the base model, a subset of 10,000 CIFAR-10 training images (1,000 for each class),
and the first 1,000 CIFAR-10 test images from Google Drive.

In [2]:
!gdown 17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes -O linf_base.pt
!gdown 1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe -O cifar10_train_10k.pt
!gdown 1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA -O cifar10_test_1k.pt

Downloading...
From: https://drive.google.com/uc?id=17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes
To: /content/linf_base.pt
100% 11.2M/11.2M [00:00<00:00, 27.1MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe
From (redirected): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe&confirm=t&uuid=09e6c88d-480e-4d8c-bac2-4fec5f4c2c40
To: /content/cifar10_train_10k.pt
100% 30.8M/30.8M [00:01<00:00, 21.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA
To: /content/cifar10_test_1k.pt
100% 3.08M/3.08M [00:00<00:00, 203MB/s]


In [3]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

NORMS = ("linf", "l2", "l1")
EPS = {"linf": 8 / 255, "l2": 0.5, "l1": 12.0}

device: cuda


In [4]:
def load_data(path):
    # The files store images as uint8. Convert them to floats in [0, 1] on the GPU.
    x, y = torch.load(path)
    return x.float().div(255).to(device), y.to(device)


# Training data: 10,000 images. Test data: the first 1,000 CIFAR-10 test images.
x_train, y_train = load_data("cifar10_train_10k.pt")
x_test, y_test = load_data("cifar10_test_1k.pt")
print("train:", tuple(x_train.shape), "test:", tuple(x_test.shape))


def augment(x):
    # Random horizontal flip and random crop (padding 4) for each image.
    n = x.shape[0]
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    padded = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 32, 1)
    j = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 1, 32)
    return padded[torch.arange(n, device=x.device).view(n, 1, 1), :, i, j].permute(0, 3, 1, 2).contiguous()


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for k in range(0, len(x), batch_size):
        j = idx[k:k + batch_size]
        yield x[j], y[j]

train: (10000, 3, 32, 32) test: (1000, 3, 32, 32)


In [5]:
# PreActResNet-18 with half width (2.8M parameters). The input is an image in [0, 1].
class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.shortcut = None
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


class PreActResNet18(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.2471, 0.2435, 0.2616]).view(1, 3, 1, 1))
        self.conv1 = nn.Conv2d(3, width, 3, 1, 1, bias=False)
        layers, in_planes = [], width
        for mult, stride in [(1, 1), (1, 1), (2, 2), (2, 1), (4, 2), (4, 1), (8, 2), (8, 1)]:
            layers.append(PreActBlock(in_planes, mult * width, stride))
            in_planes = mult * width
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_planes)
        self.linear = nn.Linear(in_planes, num_classes)

    def forward(self, x):
        out = self.layers(self.conv1((x - self.mean) / self.std))
        out = F.adaptive_avg_pool2d(F.relu(self.bn(out)), 1).flatten(1)
        return self.linear(out)


def make_model():
    return PreActResNet18().to(device)


def load_base_model():
    model = make_model()
    model.load_state_dict(torch.load("linf_base.pt", map_location=device))
    return model

# Implement the Attacks

Each attack is untargeted, uses the cross-entropy loss, and returns `x_adv` with
$\|x_{adv} - x\|_p \leq \epsilon$ and $x_{adv} \in [0, 1]$. The attacks must not change the mode (`train` or `eval`) of the model.

* `pgd_linf`: start at a uniformly random point in the $\ell_\infty$ ball. For each step, add `alpha * sign(grad)`,
  clip the perturbation to $[-\epsilon, \epsilon]$, and clip the image to $[0, 1]$.
* `pgd_l2`: start at a random point in the $\ell_2$ ball. For each step, add `alpha * grad / ||grad||_2`,
  scale the perturbation back to the $\ell_2$ ball if its norm is larger than $\epsilon$, and clip the image to $[0, 1]$.

You can use your code from HW1. The $\ell_1$ attack (sparse $\ell_1$ descent, Tramer & Boneh 2019) is complete.

In [6]:
def clip_box(x, delta):
    # Make x + delta an image in [0, 1]. This never increases |delta_i|.
    return (x + delta).clamp(0, 1) - x


def input_grad(model, x, delta, y):
    delta = delta.detach().requires_grad_(True)
    loss = F.cross_entropy(model(x + delta), y, reduction="sum")
    (grad,) = torch.autograd.grad(loss, delta)
    return grad.detach()


def l2_norm(v):
    return v.flatten(1).norm(dim=1).view(-1, 1, 1, 1)


def pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):
    # TODO: implement L-infinity PGD (use input_grad and clip_box)
    if rand_init:
        #Random start
        delta = torch.empty_like(x).uniform_(-eps, eps)
    else:
        delta = torch.zeros_like(x)
    delta = clip_box(x, delta)  #make sure x+delta is valid image

    for step in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = delta + alpha * grad.sign()
        delta = delta.clamp(-eps, eps)
        delta = clip_box(x, delta)

    x_adv = (x + delta).detach()
    return x_adv


def pgd_l2(model, x, y, eps, alpha, steps, rand_init=True):
    # TODO: implement L2 PGD (use input_grad, l2_norm and clip_box)

    if rand_init:
        delta = torch.randn_like(x)
        random_radius = torch.rand(x.shape[0], 1, 1, 1, device=x.device) * eps
        delta = delta / l2_norm(delta).clamp_min(1e-12) * random_radius
    else:
        delta = torch.zeros_like(x)
    delta = clip_box(x, delta)

    for step in range(steps):
        grad = input_grad(model, x, delta, y)
        grad_norm = l2_norm(grad).clamp_min(1e-12)
        delta = delta + alpha * grad / grad_norm

        delta_norm = l2_norm(delta).clamp_min(1e-12)
        scale_factor = (eps / delta_norm).clamp(max=1.0)
        delta = delta * scale_factor
        delta = clip_box(x, delta)

    x_adv = (x + delta).detach()
    return x_adv


In [7]:
def project_l1(delta, eps):
    # Projection of each image of delta onto the L1 ball of radius eps (Duchi et al., 2008).
    v = delta.flatten(1)
    a = v.abs()
    outside = a.sum(dim=1) > eps
    if not outside.any():
        return delta
    mu, _ = a.sort(dim=1, descending=True)
    cumsum = mu.cumsum(dim=1)
    idx = torch.arange(1, v.shape[1] + 1, device=v.device, dtype=v.dtype)
    rho = ((mu - (cumsum - eps) / idx > 0) * idx).argmax(dim=1, keepdim=True)
    theta = (cumsum.gather(1, rho) - eps) / (rho + 1).to(v.dtype)
    w = (a - theta).clamp_min(0) * v.sign()
    return torch.where(outside.unsqueeze(1), w, v).view(delta.shape)


def pgd_l1(model, x, y, eps, alpha, steps, rand_init=True, k=20):
    # Each step changes only the k coordinates with the largest gradient.
    # Coordinates at the image boundary that point outward are skipped.
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.distributions.Laplace(0.0, 1.0).sample(x.shape).to(x.device)
        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / delta.flatten(1).abs().sum(1).view(-1, 1, 1, 1) * eps * r
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y).flatten(1)
        x_cur = (x + delta).flatten(1)
        blocked = ((grad > 0) & (x_cur >= 1)) | ((grad < 0) & (x_cur <= 0))
        grad = grad.masked_fill(blocked, 0)
        top = grad.abs().topk(k, dim=1).indices
        step = torch.zeros_like(grad).scatter(1, top, grad.gather(1, top).sign())
        delta = clip_box(x, delta + alpha * (step / k).view_as(x))
        delta = project_l1(delta, eps)
    return (x + delta).detach()


ATTACKS = {"linf": pgd_linf, "l2": pgd_l2, "l1": pgd_l1}


def attack(model, x, y, norm, steps=10):
    # The attack for one norm with the default eps and step size.
    alpha = EPS["l1"] / steps ** 0.5 if norm == "l1" else 2.5 * EPS[norm] / steps
    return ATTACKS[norm](model, x, y, EPS[norm], alpha, steps)

# Evaluate Single and Multi-Norm Robust Accuracy

`test_model` gives the clean accuracy, the robust accuracy for each norm, and the union accuracy (in %) on the
1,000 test images. An image is robust for a norm if the model is correct on the clean image and on the result
of each restart of the attack. An image is robust for the union if it is robust for all three norms simultaneously.
Use the default settings (`steps=50, restarts=2`) for all the numbers in your report.

In [8]:
def test_model(model, steps=50, restarts=2, batch_size=250):
    model.eval()
    torch.manual_seed(0)  # the same random starts of the attacks for each model, so the results are comparable
    start = time.time()
    correct = {k: 0 for k in ("clean", *NORMS, "union")}
    for i in range(0, len(x_test), batch_size):
        x, y = x_test[i:i + batch_size], y_test[i:i + batch_size]
        with torch.no_grad():
            clean = model(x).argmax(1) == y
        union = clean.clone()
        for norm in NORMS:
            robust = clean.clone()
            for _ in range(restarts):
                x_adv = attack(model, x, y, norm, steps)
                with torch.no_grad():
                    robust &= model(x_adv).argmax(1) == y
            correct[norm] += robust.sum().item()
            union &= robust
        correct["clean"] += clean.sum().item()
        correct["union"] += union.sum().item()
    acc = {k: 100.0 * v / len(y_test) for k, v in correct.items()}
    print("  ".join(f"{k} {v:.1f}" for k, v in acc.items()) + f"   ({time.time() - start:.0f} s)")
    return acc

In [9]:
# The base model: L-infinity PGD adversarial training for 30 epochs on all 50,000 training images.
test_model(load_base_model())

clean 78.7  linf 48.4  l2 59.8  l1 22.4  union 22.3   (114 s)


{'clean': 78.7, 'linf': 48.4, 'l2': 59.8, 'l1': 22.4, 'union': 22.3}

# Adversarial Training

`adv_train(model, loss_fn, ...)` fine-tunes `model` for `epochs` epochs on the training subset.
For each batch, it calls `loss_fn(model, x, y, **kwargs)` and does one optimizer step on the loss.

A loss function makes the adversarial examples itself. Put the model in `eval` mode for the attack
(so that the attack does not change the BatchNorm statistics), and in `train` mode before you calculate the loss.

If `gp_beta` is set, `adv_train` does the gradient projection of RAMP (part (b)) before the optimizer step.

In [10]:
def adv_train(model, loss_fn, epochs=10, lr=0.01, batch_size=128, gp_beta=None, seed=0, **kwargs):
    torch.manual_seed(seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * (len(x_train) // batch_size))
    params = list(model.parameters())
    start = time.time()
    for epoch in range(epochs):
        losses = []
        for x, y in batches(x_train, y_train, batch_size, shuffle=True):
            if len(y) < batch_size:  # skip the last, smaller batch
                continue
            x = augment(x)
            loss = loss_fn(model, x, y, **kwargs)
            opt.zero_grad()
            if gp_beta is None:
                loss.backward()
            else:
                # RAMP (part (b)): combine the adversarial gradient g_a and the natural gradient g_n
                # for each parameter tensor. gradient_projection is defined in part (b).
                g_a = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, ga, gn in zip(params, g_a, g_n):
                    p.grad = gp_beta * gradient_projection(gn, ga) + (1 - gp_beta) * ga
            opt.step()
            sched.step()
            losses.append(loss.item())
        print(f"epoch {epoch + 1}: loss {np.mean(losses):.3f} ({time.time() - start:.0f} s)")
    model.eval()
    return model

# Part (a): Single-Norm Adversarial Training

Complete `single_norm_loss`: attack the batch with the norm `norm` (use `attack`), and return the
cross-entropy loss of the model on the adversarial images.

In [11]:
def single_norm_loss(model, x, y, norm):
    # TODO: put the model in eval mode, attack with `norm`, put the model in train mode, and return the loss
    model.eval()
    x_adv = attack(model, x, y, norm)
    model.train()
    logits_adv = model(x_adv)
    loss = F.cross_entropy(logits_adv, y)

    return loss

In [12]:
#For problem 1(a)
def standard_loss(model, x, y):
    model.train()
    logits_clean = model(x)
    loss = F.cross_entropy(logits_clean, y)
    return loss

NUM_EPOCHS_Q1 = 5

#model 1: standard model
torch.manual_seed(0)  # same random initialization for both models
model_standard = adv_train(make_model(), standard_loss, epochs=NUM_EPOCHS_Q1)

#model 2: adversarial trained
torch.manual_seed(0)
model_adv_linf = adv_train(make_model(), single_norm_loss, epochs=NUM_EPOCHS_Q1, norm="linf")

epoch 1: loss 1.875 (5 s)
epoch 2: loss 1.536 (11 s)
epoch 3: loss 1.378 (16 s)
epoch 4: loss 1.222 (21 s)
epoch 5: loss 1.132 (26 s)
epoch 1: loss 2.190 (39 s)
epoch 2: loss 2.100 (78 s)
epoch 3: loss 2.052 (118 s)
epoch 4: loss 2.030 (157 s)
epoch 5: loss 2.015 (196 s)


In [13]:
print("Standard model:")
acc_standard = test_model(model_standard)

print("adversarially trained model:")
acc_adv_linf = test_model(model_adv_linf)

Standard model:
clean 57.5  linf 0.4  l2 9.0  l1 1.6  union 0.4   (117 s)
adversarially trained model:
clean 37.4  linf 23.7  l2 29.3  l1 21.0  union 19.1   (117 s)


In [27]:
def linf_robust_accuracy(model, eps, steps=50, restarts=2, batch_size=250):
    model.eval()
    torch.manual_seed(0)
    step_size = 2.5 * eps / steps
    num_robust = 0

    for start_index in range(0, len(x_test), batch_size):
        x_batch = x_test[start_index:start_index + batch_size]
        y_batch = y_test[start_index:start_index + batch_size]

        with torch.no_grad():
            is_robust = model(x_batch).argmax(1) == y_batch

        for restart in range(restarts):
            x_adv = pgd_linf(model, x_batch, y_batch, eps, step_size, steps)
            with torch.no_grad():
                is_robust = is_robust & (model(x_adv).argmax(1) == y_batch)

        num_robust += is_robust.sum().item()

    return 100.0 * num_robust / len(x_test)


def print_eps_table(models_to_compare, eps_list):
    header = f"{'eps':>8}"
    for model_name in models_to_compare:
        header += f"  {model_name:>18}"
    print(header)

    for eps in eps_list:
        row = f"{eps * 255:>4.0f}/255"
        for model_name, model in models_to_compare.items():
            robust_acc = linf_robust_accuracy(model, eps)
            row += f"  {robust_acc:>18.1f}"
        print(row)


eps = [0, 1/255, 2/255, 4/255, 8/255, 12/255, 16/255]
models_to_compare = {"standard": model_standard,"adv linf": model_adv_linf,}
print_eps_table(models_to_compare, eps)

     eps            standard            adv linf
   0/255                57.5                37.4
   1/255                36.8                35.9
   2/255                23.4                33.5
   4/255                 6.5                29.3
   8/255                 0.4                23.7
  12/255                 0.1                17.7
  16/255                 0.0                12.8


In [25]:
def linf_attack_accuracy(model, eps, steps, step_size, rand_init, restarts, batch_size=250):
    model.eval()
    torch.manual_seed(0)
    num_correct = 0

    for start_index in range(0, len(x_test), batch_size):
        x_batch = x_test[start_index:start_index + batch_size]
        y_batch = y_test[start_index:start_index + batch_size]

        with torch.no_grad():
            is_correct = model(x_batch).argmax(1) == y_batch

        for restart in range(restarts):
            x_adv = pgd_linf(model, x_batch, y_batch, eps, step_size, steps, rand_init=rand_init)
            with torch.no_grad():
                is_correct = is_correct & (model(x_adv).argmax(1) == y_batch)

        num_correct += is_correct.sum().item()

    return 100.0 * num_correct / len(x_test)


def pgd_accuracy(model, eps, steps=50, restarts=2):
    step_size = 2.5 * eps / steps
    return linf_attack_accuracy(model, eps, steps, step_size, rand_init=True, restarts=restarts)


def fgsm_accuracy(model, eps):
    return linf_attack_accuracy(model, eps, steps=1, step_size=eps, rand_init=False, restarts=1)


EPS_LIST_Q1B = [1 / 255, 2 / 255, 4 / 255, 8 / 255, 12 / 255, 16 / 255]
models_q1b = {
    "standard": model_standard,
    "adv linf": model_adv_linf,
}

#results_q1b[model_name]["clean"] -> clean accuracy
#results_q1b[model_name]["pgd"][eps]-> accuracy PGD
#results_q1b[model_name]["fgsm"][eps] -> accuracy FGSM
results_q1b = {}
for model_name, model in models_q1b.items():
    model.eval()
    with torch.no_grad():
        clean_accuracy = 100.0 * (model(x_test).argmax(1) == y_test).float().mean().item()

    results_q1b[model_name] = {"clean": clean_accuracy, "pgd": {}, "fgsm": {}}
    for eps in EPS_LIST_Q1B:
        results_q1b[model_name]["pgd"][eps] = pgd_accuracy(model, eps)
        results_q1b[model_name]["fgsm"][eps] = fgsm_accuracy(model, eps)

In [26]:
# Table for accuracy under each attack, and clean accuracy - attacked accuracy.
column_names = ["std PGD", "std FGSM", "adv PGD", "adv FGSM"]

for model_name in models_q1b:
    print(f"clean accuracy of {model_name}: {results_q1b[model_name]['clean']:.1f}")

for table_title, show_drop in [("Accuracy under attack", False), ("Accuracy drop (clean-attacked)", True)]:
    print()
    print(table_title)
    header = f"{'eps':>8}"
    for column_name in column_names:
        header += f"  {column_name:>9}"
    print(header)

    for eps in EPS_LIST_Q1B:
        row = f"{eps*255:>4.0f}/255"
        for model_name in ["standard", "adv linf"]:
            for attack_name in ["pgd", "fgsm"]:
                value = results_q1b[model_name][attack_name][eps]
                if show_drop:
                    value = results_q1b[model_name]["clean"] - value
                row += f"  {value:>9.1f}"
        print(row)

clean accuracy of standard: 57.5
clean accuracy of adv linf: 37.4

Accuracy under attack
     eps    std PGD   std FGSM    adv PGD   adv FGSM
   1/255       36.8       38.8       35.9       35.9
   2/255       23.4       27.5       33.5       33.6
   4/255        6.5       12.8       29.3       29.4
   8/255        0.4        3.1       23.7       24.3
  12/255        0.1        1.0       17.7       19.2
  16/255        0.0        0.4       12.8       15.0

Accuracy drop (clean-attacked)
     eps    std PGD   std FGSM    adv PGD   adv FGSM
   1/255       20.7       18.7        1.5        1.5
   2/255       34.1       30.0        3.9        3.8
   4/255       51.0       44.7        8.1        8.0
   8/255       57.1       54.4       13.7       13.1
  12/255       57.4       56.5       19.7       18.2
  16/255       57.5       57.1       24.6       22.4


In [15]:
"""
model_linf = adv_train(load_base_model(), single_norm_loss, norm="linf")
test_model(model_linf)
"""

'\nmodel_linf = adv_train(load_base_model(), single_norm_loss, norm="linf")\ntest_model(model_linf)\n'

In [16]:
"""
model_l2 = adv_train(load_base_model(), single_norm_loss, norm="l2")
test_model(model_l2)
"""

'\nmodel_l2 = adv_train(load_base_model(), single_norm_loss, norm="l2")\ntest_model(model_l2)\n'

In [17]:
"""
model_l1 = adv_train(load_base_model(), single_norm_loss, norm="l1")
test_model(model_l1)
"""

'\nmodel_l1 = adv_train(load_base_model(), single_norm_loss, norm="l1")\ntest_model(model_l1)\n'

# Part (b): Multi-Norm Adversarial Training (RAMP)

* `ramp_loss`: attack with $q = \ell_\infty$ and $r = \ell_1$. The loss is the mean of
  $\max(\mathcal{L}_{CE}(x^q), \mathcal{L}_{CE}(x^r))$ plus `lam` times $\mathcal{L}_{KL} = KL(p_q \| p_r)$.
  Calculate $\mathcal{L}_{KL}$ only on the images that the model classifies correctly under the $\ell_q$ attack.
* `gradient_projection(g_n, g_a)`: $\cos(g_n, g_a) \cdot g_n$ if $\cos(g_n, g_a) > 0$, else $0$.

`adv_train(..., gp_beta=0.5)` then uses `p.grad = gp_beta * GP(g_n, g_a) + (1 - gp_beta) * g_a` for each parameter
tensor, where $g_a$ is the gradient of `ramp_loss` and $g_n$ is the gradient of the cross-entropy loss on the clean images.

In [18]:
"""
def ramp_loss(model, x, y, lam=0.5):
    # TODO: in eval mode, attack with linf and l1; in train mode, return the max of the two losses plus lam * KL
    raise NotImplementedError


def gradient_projection(g_n, g_a):
    # TODO: GP(g_n, g_a) for one parameter tensor
    raise NotImplementedError
"""

'\ndef ramp_loss(model, x, y, lam=0.5):\n    # TODO: in eval mode, attack with linf and l1; in train mode, return the max of the two losses plus lam * KL\n    raise NotImplementedError\n\n\ndef gradient_projection(g_n, g_a):\n    # TODO: GP(g_n, g_a) for one parameter tensor\n    raise NotImplementedError\n'

In [19]:
"""
model_ramp = adv_train(load_base_model(), ramp_loss, gp_beta=0.5)
test_model(model_ramp)
"""

'\nmodel_ramp = adv_train(load_base_model(), ramp_loss, gp_beta=0.5)\ntest_model(model_ramp)\n'

# Part (c): Your Method

Write the training loss of your method in `my_loss`, and train with `adv_train(load_base_model(), my_loss)`.
If your method needs a different update (not the gradient of one loss), write your own copy of `adv_train`.
Run your method and RAMP with `seed=0` and `seed=1`.
Then evaluate your best model again with stronger attacks: `test_model(model, steps=200, restarts=5)`.

In [20]:
def my_loss(model, x, y):
    # TODO: your method
    raise NotImplementedError

In [21]:

# model_mine = adv_train(load_base_model(), my_loss)
# test_model(model_mine)
